# Generate masks

Step 1: read and validate the acquisition manifest.
Step 2: select a test subset.
Step 3: use a shorter Dynamic World time window.
Step 4: apply a scene preset.
Step 5: combine Dynamic World mean scores with majority label guidance.
Step 6: clean the raw mask.
Step 7: save the mask, split preview, and mask manifest.


In [115]:
from pathlib import Path
from urllib.request import urlopen
import shutil
import sys

import ee
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio as rio
from matplotlib.colors import ListedColormap

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable, **kwargs):
        return iterable

ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "data" / "manifests").exists() and (candidate / "notebooks").exists():
        ROOT = candidate
        break

if ROOT is None:
    raise FileNotFoundError("Could not find project root with data/manifests and notebooks")

PACKAGE_ROOT = ROOT / "src"
if (PACKAGE_ROOT / "land_cover").exists():
    sys.path.insert(0, str(PACKAGE_ROOT))
    from land_cover import DIRECTORIES, mask_name, sample_dir, viz_name
else:
    DIRECTORIES = {
        "data_samples_prepared": ROOT / "data" / "samples_prepared",
        "data_samples_generated": ROOT / "data" / "samples_generated",
        "data_manifests": ROOT / "data" / "manifests",
    }

    def sample_dir(sample: str, root_key: str = "data_samples_generated"):
        return DIRECTORIES[root_key] / sample

    def mask_name(sample: str) -> str:
        return f"{sample}_Mask.tif"

    def viz_name(sample: str) -> str:
        return f"{sample}_viz.png"

MANIFEST_PATH = DIRECTORIES["data_manifests"] / "gee_acquisition_manifest.csv"
MASK_MANIFEST_PATH = DIRECTORIES["data_manifests"] / "mask_manifest.csv"

try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()


In [116]:
PROFILE_PATH = ROOT / "location_label_profiles_5.csv"

REQUIRED_COLUMNS = [
    "sample_name",
    "latitude",
    "longitude",
    "start_date",
    "end_date",
    "buffer_meters",
    "chip_size",
    "spectral_path",
]

samples = pd.read_csv(MANIFEST_PATH)

missing = [col for col in REQUIRED_COLUMNS if col not in samples.columns]
if missing:
    raise ValueError(f"Missing manifest columns: {missing}")

if "name" not in samples.columns:
    samples["name"] = samples["sample_name"]

if "terrain_focus" not in samples.columns:
    samples["terrain_focus"] = ""

profiles = pd.read_csv(PROFILE_PATH)[["name", "profile_code_5", "profile_name_5"]].drop_duplicates(subset=["name"])
samples = samples.merge(profiles, on="name", how="left")
samples["profile_code_5"] = samples["profile_code_5"].fillna("G0")
samples["profile_name_5"] = samples["profile_name_5"].fillna("Default")

keep = REQUIRED_COLUMNS + ["name", "terrain_focus", "profile_code_5", "profile_name_5"]
samples = samples[keep].drop_duplicates(subset=["sample_name"]).copy()
samples["spectral_exists"] = samples["spectral_path"].map(lambda p: (ROOT / p).exists())

available = samples[samples["spectral_exists"]].copy()
if available.empty:
    raise FileNotFoundError("No spectral chips found in data/samples_generated. Run notebooks/data_acquisition.ipynb first.")

samples


,sample_name,latitude,longitude,start_date,end_date,buffer_meters,chip_size,spectral_path,name,terrain_focus,profile_code_5,profile_name_5,spectral_exists
0,CairoUniv,30.026330,31.209090,2025-05-01,2025-08-28,1280,256,data/samples_generated/CairoUniv/CairoUniv_Spe...,Cairo University,urban/infrastructure,G1,Urban / Urban–Desert,True
1,IconicTower,30.010918,31.701837,2025-05-01,2025-08-28,1280,256,data/samples_generated/IconicTower/IconicTower...,"Iconic Tower, New Administrative Capital",urban/infrastructure,G1,Urban / Urban–Desert,True
2,SiwaOasis,29.203200,25.519500,2025-05-01,2025-08-28,1280,256,data/samples_generated/SiwaOasis/SiwaOasis_Spe...,Siwa Oasis,oasis/desert,G3,Oasis / Reclamation / Agriculture–Desert,True
3,KarnakLuxor,25.718000,32.658300,2025-05-01,2025-08-28,1280,256,data/samples_generated/KarnakLuxor/KarnakLuxor...,"Karnak, Luxor",nile/urban,G2,Nile / Delta / Canal / Wetland,True
4,PhilaeAswan,24.025500,32.884400,2025-05-01,2025-08-28,1280,256,data/samples_generated/PhilaeAswan/PhilaeAswan...,"Philae, Aswan",water/rocky urban edge,G5,Red Sea / Rocky / Open-Water Desert,True
5,HawaraFayoum,29.274700,30.898600,2025-05-01,2025-08-28,1280,256,data/samples_generated/HawaraFayoum/HawaraFayo...,"Hawara, Fayoum",agriculture/desert edge,G3,Oasis / Reclamation / Agriculture–Desert,True
6,Alexandria,31.205753,29.924526,2025-05-01,2025-08-28,1280,256,data/samples_generated/Alexandria/Alexandria_S...,Alexandria,coastal urban,G4,Mediterranean Coast / Lagoon / Sand / Port,True
7,GreatPyramidOfGiza,29.974167,31.133947,2025-05-01,2025-08-28,1280,256,data/samples_generated/GreatPyramidOfGiza/Grea...,Great Pyramid of Giza,urban/desert edge,G1,Urban / Urban–Desert,True
8,6thOfOctoberCity,29.816667,31.050000,2025-05-01,2025-08-28,1280,256,data/samples_generated/6thOfOctoberCity/6thOfO...,6th of October City,urban/desert edge,G1,Urban / Urban–Desert,True
9,NewCairo,30.030000,31.470000,2025-05-01,2025-08-28,1280,256,data/samples_generated/NewCairo/NewCairo_Spect...,New Cairo,urban/desert edge,G1,Urban / Urban–Desert,True


In [117]:
SAMPLE_LIMIT = 12
SAMPLE_NAMES = []
WINDOW_DAYS = 30
OVERWRITE = True

DEFAULT_PARAMS = {
    "unknown_max_size": 64,
    "green_bonus": 0.18,
    "sand_bonus": 0.00,
    "water_bonus": 0.20,
    "cement_bonus": 0.62,
    "water_weight": 1.15,
    "min_score": 0.12,
    "water_keep": 0.32,
    "water_bridge": 1,
    "green_keep": 0.06,
    "green_ratio": 0.40,
    "cement_keep": 0.08,
    "cement_ratio": 0.36,
    "ndvi_keep": 0.30,
    "gray_keep": 0.17,
    "cement_floor": 0.07,
    "cement_reclaim_ndbi": 1.00,
    "green_reclaim_ndvi": 1.00,
}

PROFILE_PARAMS = {
    "G1": {"green_bonus": 0.30, "cement_bonus": 0.42, "green_ratio": 0.28, "cement_ratio": 0.48, "ndvi_keep": 0.22, "gray_keep": 0.14, "cement_floor": 0.10, "cement_reclaim_ndbi": 0.08, "green_reclaim_ndvi": 0.30},
    "G2": {"green_bonus": 0.24, "water_bonus": 0.28, "cement_bonus": 0.48, "water_weight": 1.28, "water_keep": 0.24, "water_bridge": 2, "green_ratio": 0.34, "ndvi_keep": 0.24},
    "G3": {"green_bonus": 0.28, "water_bonus": 0.16, "cement_bonus": 0.40, "water_weight": 1.20, "min_score": 0.10, "green_ratio": 0.30, "ndvi_keep": 0.22},
    "G4": {"water_bonus": 0.30, "cement_bonus": 0.56, "water_weight": 1.30, "water_keep": 0.24, "water_bridge": 2, "cement_ratio": 0.34, "gray_keep": 0.18},
    "G5": {"green_bonus": 0.12, "water_bonus": 0.32, "cement_bonus": 0.44, "water_weight": 1.35, "water_keep": 0.22, "water_bridge": 2, "green_ratio": 0.45, "cement_ratio": 0.40, "ndvi_keep": 0.34},
}

def params_for(row):
    params = DEFAULT_PARAMS.copy()
    params.update(PROFILE_PARAMS.get(getattr(row, "profile_code_5", "G0"), {}))
    return params

RGB_P_LOW = 2
RGB_P_HIGH = 98
RGB_GAMMA = 1.15

if SAMPLE_NAMES:
    selected = available[available["sample_name"].isin(SAMPLE_NAMES)].copy()
elif SAMPLE_LIMIT is not None:
    selected = available.head(SAMPLE_LIMIT).copy()
else:
    selected = available.copy()

selected[["sample_name", "profile_code_5", "profile_name_5", "terrain_focus"]]


,sample_name,profile_code_5,profile_name_5,terrain_focus
0,CairoUniv,G1,Urban / Urban–Desert,urban/infrastructure
1,IconicTower,G1,Urban / Urban–Desert,urban/infrastructure
2,SiwaOasis,G3,Oasis / Reclamation / Agriculture–Desert,oasis/desert
3,KarnakLuxor,G2,Nile / Delta / Canal / Wetland,nile/urban
4,PhilaeAswan,G5,Red Sea / Rocky / Open-Water Desert,water/rocky urban edge
5,HawaraFayoum,G3,Oasis / Reclamation / Agriculture–Desert,agriculture/desert edge
6,Alexandria,G4,Mediterranean Coast / Lagoon / Sand / Port,coastal urban
7,GreatPyramidOfGiza,G1,Urban / Urban–Desert,urban/desert edge
8,6thOfOctoberCity,G1,Urban / Urban–Desert,urban/desert edge
9,NewCairo,G1,Urban / Urban–Desert,urban/desert edge


In [118]:
DW_BANDS = [
    "water",
    "trees",
    "grass",
    "flooded_vegetation",
    "crops",
    "shrub_and_scrub",
    "built",
    "bare",
    "snow_and_ice",
]

S2_BANDS = ["B2", "B3", "B4", "B8", "B11"]
PROJECT_CLASS_IDS = {0: "unknown", 1: "greenery", 2: "sand", 3: "water", 4: "cement"}
PROJECT_BANDS = ["greenery", "sand", "water", "cement"]
LABEL_GROUPS = {
    "greenery": [1, 2, 4],
    "sand": [7],
    "water": [0],
    "cement": [6],
}
MASK_COLORS = ["#000000", "#008a1a", "#d7a400", "#4217b5", "#8f8f8f"]
MASK_CMAP = ListedColormap(MASK_COLORS)
def date_window(row):
    start = pd.Timestamp(row.start_date)
    end = pd.Timestamp(row.end_date)
    mid = start + (end - start) / 2
    half = pd.Timedelta(days=WINDOW_DAYS // 2)
    return (mid - half).strftime("%Y-%m-%d"), (mid + half).strftime("%Y-%m-%d")


def build_region(row):
    return ee.Geometry.Point([row.longitude, row.latitude]).buffer(row.buffer_meters).bounds()


def mode_bonus(label_mode, ids, value, name):
    hit = label_mode.eq(ids[0])
    for class_id in ids[1:]:
        hit = hit.Or(label_mode.eq(class_id))
    return hit.multiply(value).rename(name)


def dynamic_world_stats(row):
    region = build_region(row)
    start, end = date_window(row)
    coll = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(region).filterDate(start, end)
    probs = coll.select(DW_BANDS)
    labels = coll.select("label")
    s2 = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(region)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
        .select(S2_BANDS)
    )
    empty_probs = ee.Image.constant([0] * len(DW_BANDS)).rename(DW_BANDS)
    empty_label = ee.Image.constant(255).rename("label").toUint8()
    empty_s2 = ee.Image.constant([0] * len(S2_BANDS)).rename(S2_BANDS)
    mean_probs = ee.Image(ee.Algorithms.If(probs.size().gt(0), probs.mean(), empty_probs)).clip(region)
    label_mode = ee.Image(ee.Algorithms.If(labels.size().gt(0), labels.mode(), empty_label)).clip(region)
    median_s2 = ee.Image(ee.Algorithms.If(s2.size().gt(0), s2.median(), empty_s2)).clip(region)
    return region, coll, mean_probs, label_mode, median_s2, start, end


In [119]:
def project_mask(row, params):
    region, dw, probs, label_mode, s2, short_start, short_end = dynamic_world_stats(row)

    base = ee.Image.cat([
        probs.select(["trees", "grass", "crops", "flooded_vegetation"]).reduce(ee.Reducer.sum()).rename("greenery"),
        probs.select(["bare", "shrub_and_scrub"]).reduce(ee.Reducer.sum()).rename("sand"),
        probs.select("water").multiply(params["water_weight"]).rename("water"),
        probs.select("built").rename("cement"),
    ]).rename(PROJECT_BANDS)

    scores = ee.Image.cat([
        base.select("greenery").add(mode_bonus(label_mode, LABEL_GROUPS["greenery"], params["green_bonus"], "greenery")),
        base.select("sand").add(mode_bonus(label_mode, LABEL_GROUPS["sand"], params["sand_bonus"], "sand")),
        base.select("water").add(mode_bonus(label_mode, LABEL_GROUPS["water"], params["water_bonus"], "water")),
        base.select("cement").add(mode_bonus(label_mode, LABEL_GROUPS["cement"], params["cement_bonus"], "cement")),
    ]).rename(PROJECT_BANDS)

    mapped_total = scores.reduce(ee.Reducer.sum()).rename("mapped_total")
    mask = scores.toArray().arrayArgmax().arrayGet([0]).add(1).rename("mask").toUint8()
    mask = mask.where(mapped_total.lt(params["min_score"]), 0)

    if params["water_bridge"] > 0:
        water = (
            scores.select("water")
            .gt(params["water_keep"])
            .focalMax(radius=params["water_bridge"], units="pixels", kernelType="square")
            .focalMin(radius=params["water_bridge"], units="pixels", kernelType="square")
        )
        mask = mask.where(water, 3)

    ndvi = s2.normalizedDifference(["B8", "B4"]).rename("ndvi")
    ndbi = s2.normalizedDifference(["B11", "B8"]).rename("ndbi")
    gray = (
        s2.select(["B4", "B3", "B2"]).reduce(ee.Reducer.stdDev())
        .divide(s2.select(["B4", "B3", "B2"]).reduce(ee.Reducer.mean()).add(1e-6))
        .rename("gray")
    )

    greenery = scores.select("greenery")
    cement = scores.select("cement")
    sand = scores.select("sand")
    green_mode = label_mode.eq(1).Or(label_mode.eq(2)).Or(label_mode.eq(4))
    cement_mode = label_mode.eq(6)
    green_pull = greenery.gt(params["green_keep"]).And(greenery.gte(sand.multiply(params["green_ratio"])))
    green_pull = green_pull.Or(green_mode.And(greenery.gt(params["green_keep"] * 0.5)))
    green_pull = green_pull.Or(mask.eq(2).And(ndvi.gt(params["ndvi_keep"])))
    green_pull = green_pull.Or(mask.eq(4).And(ndvi.gt(params["green_reclaim_ndvi"])))
    mask = mask.where(green_pull.And(mask.neq(3)), 1)
    cement_pull = cement.gt(params["cement_keep"]).And(cement.gte(sand.multiply(params["cement_ratio"])))
    cement_pull = cement_pull.Or(cement_mode.And(cement.gt(params["cement_keep"] * 0.5)))
    cement_pull = cement_pull.Or(mask.eq(2).And(gray.lt(params["gray_keep"])).And(cement.gt(params["cement_floor"])).And(ndvi.lt(params["ndvi_keep"])))
    cement_pull = cement_pull.Or(mask.eq(2).And(ndbi.gt(params["cement_reclaim_ndbi"])).And(cement.gt(params["cement_floor"])).And(ndvi.lt(params["ndvi_keep"])))
    mask = mask.where(cement_pull.And(mask.neq(3)), 4)

    return region, dw, scores, mapped_total, mask.clip(region), short_start, short_end


def clean_mask(mask, unknown_max_size):
    cleaned = mask.rename("mask").toUint8()
    if unknown_max_size <= 0:
        return cleaned

    fill = (
        cleaned.updateMask(cleaned.gt(0))
        .focalMode(radius=1, units="pixels", kernelType="square")
        .unmask(0)
        .rename("fill")
        .toUint8()
    )
    unknown = cleaned.eq(0)
    small_unknown = (
        unknown.selfMask()
        .connectedPixelCount(maxSize=unknown_max_size + 1, eightConnected=True)
        .lte(unknown_max_size)
    )
    return cleaned.where(unknown.And(small_unknown), fill).rename("mask").toUint8()


In [120]:
def export_mask(image, output_path, region, chip_size):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    params = {
        "name": output_path.stem,
        "bands": ["mask"],
        "region": region,
        "dimensions": [int(chip_size), int(chip_size)],
        "format": "GEO_TIFF",
        "filePerBand": False,
    }
    url = image.rename("mask").toUint8().getDownloadURL(params)
    with urlopen(url) as response, output_path.open("wb") as file_obj:
        shutil.copyfileobj(response, file_obj)


def percentile_stretch(channel):
    low, high = np.percentile(channel, [RGB_P_LOW, RGB_P_HIGH])
    high = max(high, low + 1e-6)
    scaled = np.clip((channel - low) / (high - low), 0, 1)
    return np.power(scaled, 1 / RGB_GAMMA)


def load_rgb(path):
    with rio.open(path) as src:
        rgb = src.read([4, 3, 2]).astype(np.float32)
    rgb = np.moveaxis(rgb, 0, -1)
    return np.stack([percentile_stretch(rgb[..., i]) for i in range(3)], axis=-1)


def load_mask(path):
    with rio.open(path) as src:
        return src.read(1).astype(np.uint8)


def save_preview(spectral_path, mask_path, output_path):
    rgb = load_rgb(spectral_path)
    mask = load_mask(mask_path)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5), facecolor="white")
    axes[0].imshow(rgb)
    axes[0].set_title("RGB")
    axes[1].imshow(mask, cmap=MASK_CMAP, vmin=0, vmax=4, interpolation="nearest")
    axes[1].set_title("Mask")

    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])

    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def mask_stats(path):
    mask = load_mask(path)
    total = mask.size
    return {f"{name}_share": round(float((mask == class_id).sum() / total), 4) for class_id, name in PROJECT_CLASS_IDS.items()}


def save_manifest(records):
    current = pd.DataFrame(records)
    if MASK_MANIFEST_PATH.exists():
        previous = pd.read_csv(MASK_MANIFEST_PATH)
        if "sample_name" in previous.columns:
            previous = previous[~previous["sample_name"].isin(current["sample_name"])]
            current = pd.concat([previous, current], ignore_index=True)
    current = current.sort_values("sample_name").reset_index(drop=True)
    current.to_csv(MASK_MANIFEST_PATH, index=False)
    return current


In [121]:
records = []

for row in tqdm(selected.itertuples(index=False), total=len(selected), desc="Masks"):
    sample_root = sample_dir(row.sample_name)
    sample_root.mkdir(parents=True, exist_ok=True)

    spectral_path = ROOT / row.spectral_path
    mask_path = sample_root / mask_name(row.sample_name)
    preview_path = sample_root / viz_name(row.sample_name)
    prepared_mask_path = DIRECTORIES["data_samples_prepared"] / row.sample_name / mask_name(row.sample_name)

    if not spectral_path.exists():
        records.append({"sample_name": row.sample_name, "status": "missing_spectral"})
        continue

    params = params_for(row)
    region, dw, scores, mapped_total, mask, short_start, short_end = project_mask(row, params)
    cleaned = clean_mask(mask, params["unknown_max_size"])
    dw_images = dw.size().getInfo()

    if not OVERWRITE and mask_path.exists() and preview_path.exists():
        status = "skipped"
    else:
        export_mask(cleaned, mask_path, region, row.chip_size)
        save_preview(spectral_path, mask_path, preview_path)
        status = "saved"

    stats = mask_stats(mask_path)
    ref_stats = {}
    if prepared_mask_path.exists():
        ref_stats = {f"ref_{k}": v for k, v in mask_stats(prepared_mask_path).items()}

    records.append({
        "sample_name": row.sample_name,
        "profile_code_5": row.profile_code_5,
        "profile_name_5": row.profile_name_5,
        "short_start": short_start,
        "short_end": short_end,
        "window_days": WINDOW_DAYS,
        **params,
        "spectral_path": str(spectral_path.relative_to(ROOT)),
        "mask_path": str(mask_path.relative_to(ROOT)),
        "viz_path": str(preview_path.relative_to(ROOT)),
        **stats,
        **ref_stats,
        "dw_images": dw_images,
        "status": status,
    })

manifest = save_manifest(records)
(
    manifest[manifest["sample_name"].isin(selected["sample_name"])]
    [["sample_name", "spectral_path", "mask_path"]]
    .reset_index(drop=True)
)


Masks: 100%|██████████| 12/12 [00:49<00:00,  4.12s/it]


,sample_name,spectral_path,mask_path
0,6thOfOctoberCity,data/samples_generated/6thOfOctoberCity/6thOfO...,data/samples_generated/6thOfOctoberCity/6thOfO...
1,Alexandria,data/samples_generated/Alexandria/Alexandria_S...,data/samples_generated/Alexandria/Alexandria_M...
2,CairoUniv,data/samples_generated/CairoUniv/CairoUniv_Spe...,data/samples_generated/CairoUniv/CairoUniv_Mas...
3,EastPortSaidPort,data/samples_generated/EastPortSaidPort/EastPo...,data/samples_generated/EastPortSaidPort/EastPo...
4,GreatPyramidOfGiza,data/samples_generated/GreatPyramidOfGiza/Grea...,data/samples_generated/GreatPyramidOfGiza/Grea...
5,HawaraFayoum,data/samples_generated/HawaraFayoum/HawaraFayo...,data/samples_generated/HawaraFayoum/HawaraFayo...
6,IconicTower,data/samples_generated/IconicTower/IconicTower...,data/samples_generated/IconicTower/IconicTower...
7,KarnakLuxor,data/samples_generated/KarnakLuxor/KarnakLuxor...,data/samples_generated/KarnakLuxor/KarnakLuxor...
8,NewCairo,data/samples_generated/NewCairo/NewCairo_Spect...,data/samples_generated/NewCairo/NewCairo_Mask.tif
9,PhilaeAswan,data/samples_generated/PhilaeAswan/PhilaeAswan...,data/samples_generated/PhilaeAswan/PhilaeAswan...
